### imports and pandas settings

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

In [2]:
pd.set_option("display.max_rows", None)

### set the path to data file

In [3]:
load_dotenv()

CF_OUTPUTS = Path(os.getenv("CF_OUTPUTS"))
print(CF_OUTPUTS)
print(CF_OUTPUTS.is_dir())

/home/dyretna/Dokument/Code/GitHub/nightingale_projects/cf_bench/cf_outputs
True


In [6]:
batch_data = CF_OUTPUTS / "SMOTE" / "gridsearched_predictors" / "XGBoost_thres0.5_2026-05-07" / "annotated.csv"

### load data 
convert to string to fillna with whitespace
we drop:  
- target_risk (half of "risk_before") , and 
- "hltprhc"

we rename "original" in df_id to "xin" (HL standards)

In [7]:
# Load CSV
batch_df = pd.read_csv(batch_data)

In [8]:
# Feature columns
feature_cols = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "bmi", "dosprt"]

# dtypes
int_columns = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "dosprt", "hltprhc", "Nchanged"]
float_columns = ["bmi", "gower_distance", "risk_before", "predicted_risk_after", "cf_gen_time_sec"]


# Convert and fill NaN with empty strings
batch_df[int_columns] = batch_df[int_columns].astype("Int64").astype("string").fillna("")
batch_df[float_columns] = batch_df[float_columns].round(4).astype("string").fillna("")
batch_df["valid"] = batch_df["valid"].astype("string").fillna("")

In [9]:
batch_df = batch_df.drop(columns=["hltprhc", "target_risk"])
batch_df["cf_id"] = batch_df["cf_id"].replace({"original": "xin"})

### All CFs

In [10]:
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,1.12,,,,,
1,0,cf_1,,,,,,,18.4191,,,0.8999,1,False,0.2114,0.217
2,0,cf_2,3,,,,,,18.2326,,,0.9081,2,False,0.2114,0.2744
3,0,cf_3,,,6,,,,16.7843,,,0.9717,2,False,0.2114,0.1366
4,0,cf_4,,,,,,,16.1409,5,,1.0,2,False,0.2114,0.114
5,0,cf_5,,,,7,,,18.9745,7,,0.8755,3,False,0.2114,0.1971
6,0,cf_6,,,5,,,,18.9619,2,,0.8761,3,True,0.2114,0.0713
7,0,cf_7,3,,,,,,18.9619,3,,0.8761,3,False,0.2114,0.1272
8,0,cf_8,3,,,,,,18.9619,7,,0.8761,3,False,0.2114,0.3297
9,0,cf_9,,,6,,,,18.9565,2,,0.8763,3,True,0.2114,0.0839


### Filtering Valid CFs

In [11]:
from cf_bench.utils import filter_valid_cfs

batch_df = filter_valid_cfs(batch_df)
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,1.12,,,,,
9,0,cf_6,,,5,,,,18.9619,2,,0.8761,3,True,0.2114,0.0713
10,0,cf_9,,,6,,,,18.9565,2,,0.8763,3,True,0.2114,0.0839
1,1,xin,3,4,1,2,3,0,22.3757,0,1.08,,,,,
11,1,cf_5,,,,3,1,,22.3757,,,0.8796,3,True,0.5611,0.2776
12,1,cf_7,,,,,1,,22.3757,1,,0.8796,3,True,0.5611,0.1843
13,1,cf_10,,,4,,1,,22.3757,7,,0.8796,4,True,0.5611,0.1848
2,2,xin,5,3,1,1,4,0,22.694,7,1.28,,,,,
14,2,cf_4,,,,4,2,,22.6757,,,0.8754,3,True,0.4676,0.2217
15,2,cf_6,,,2,7,,,22.6757,,,0.8754,3,True,0.4676,0.1629


### select one CF

In [12]:
from cf_bench.utils import select_one_cf_per_query

# Select one CF per observation (prefers valid CFs without violations)
single_cf_df = select_one_cf_per_query(batch_df, prefer_no_violations=True)
single_cf_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,1.12,,,,,
9,0,cf_9,,,6,,,,18.9565,2,,0.8763,3,True,0.2114,0.0839
1,1,xin,3,4,1,2,3,0,22.3757,0,1.08,,,,,
10,1,cf_5,,,,3,1,,22.3757,,,0.8796,3,True,0.5611,0.2776
2,2,xin,5,3,1,1,4,0,22.694,7,1.28,,,,,
11,2,cf_6,,,2,7,,,22.6757,,,0.8754,3,True,0.4676,0.1629
3,3,xin,3,3,6,6,2,0,24.3418,1,1.28,,,,,
12,3,cf_1,,,,,,,24.3375,,,0.8752,1,False,0.4413,0.4413
4,4,xin,5,4,2,7,1,0,21.2585,3,1.33,,,,,
13,4,cf_1,,,,,,,21.2585,,,0.9868,1,False,0.0502,0.0502


In [13]:
from cf_bench.dice_adapters import DiceRecommender

recommender = DiceRecommender()

# Format a single query
for idx in range(0, 10):
    print(recommender.format_query(batch_df, query_index=idx))


=== Query index '0' ===
Task / Target: hltprhc
Query instance index: 0

Original instance:
  etfruit: 4
  eatveg: 3
  cgtsmok: 3
  alcfreq: 4
  slprl: 2
  paccnois: 0
  bmi: 18.9866
  dosprt: 0


=== Counterfactuals ===

--- cf_6 ---
Predicted risk: 0.0713
Valid: True
Changes:
  cgtsmok: 3 → 5
  bmi: 18.9866 → 18.9619
  dosprt: 0 → 2

--- cf_9 ---
Predicted risk: 0.0839
Valid: True
Changes:
  cgtsmok: 3 → 6
  bmi: 18.9866 → 18.9565
  dosprt: 0 → 2


=== Query index '1' ===
Task / Target: hltprhc
Query instance index: 1

Original instance:
  etfruit: 3
  eatveg: 4
  cgtsmok: 1
  alcfreq: 2
  slprl: 3
  paccnois: 0
  bmi: 22.3757
  dosprt: 0


=== Counterfactuals ===

--- cf_5 ---
Predicted risk: 0.2776
Valid: True
Changes:
  alcfreq: 2 → 3
  slprl: 3 → 1

--- cf_7 ---
Predicted risk: 0.1843
Valid: True
Changes:
  slprl: 3 → 1
  dosprt: 0 → 1

--- cf_10 ---
Predicted risk: 0.1848
Valid: True
Changes:
  cgtsmok: 1 → 4
  slprl: 3 → 1
  dosprt: 0 → 7


=== Query index '2' ===
Task / Target